# 🚀 AWARE: AI-Based Workplace Assessment for Readiness and Safety
**Deep Learning Fatigue Detection System (Deteksi Wajah Lelah & Mengantuk)**

--- 
Notebook ini berisi keseluruhan alur (End-to-End Pipeline) pengembangan model Deep Learning untuk deteksi kelelahan (fatigue) pada karyawan sebelum memulai shift kerja. 

### 🎯 Highlight Fitur Utama (Sesuai Tugas & Requirement Profesional):
1. **Model Architecture**: Menggunakan **Model Subclassing** (`tf.keras.Model`) dengan backbone **MobileNetV2** (pre-trained ImageNet) yang ringan dan cepat.
2. **Custom Layer**: Implementasi **Channel Attention (Squeeze-and-Excitation Block)** untuk memfokuskan ekstraksi fitur pada area penting wajah (mata berat, mata tertutup, ekspresi lelah, mulut menguap).
3. **Custom Loss Function**: Menggunakan **Binary Focal Loss** dengan **Label Smoothing** untuk mengatasi masalah *class imbalance* (3.3x rasio imbalance pada dataset).
4. **Custom Training Loop**: Menggunakan **`tf.GradientTape`** secara manual dengan strategi training 2 fase (Phase 1: Frozen Backbone, Phase 2: Fine-Tuning).
5. **Custom Callback & Monitoring**: Integrasi penuh dengan **TensorBoard** (loss, accuracy, MAE log) serta penyimpanan model terbaik (*best checkpoint*).
6. **Export Model**: Siap deploy dalam format **`.keras`**, **`SavedModel`**, dan **`TFLite`**.

---

## 📁 Bagian 1: Setup Environment Local & Library
Pastikan environment Python yang digunakan adalah Python 3.10 / 3.11 lokal (misalnya `C:\Users\dzikri\AppData\Local\Programs\Python\Python311`).

In [ ]:
import os
import sys
import time
import logging
from pathlib import Path

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# Menambahkan root directory project ke path agar modul src & configs dapat diimpor
ROOT = Path(os.getcwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from configs.config import *
from src.utils import setup_logging, configure_device, ensure_directories

# Konfigurasi direktori & logging
ensure_directories()
setup_logging(log_level="INFO")
logger = logging.getLogger("AWARE_Notebook")

# Pengecekan Device & Mixed Precision
device = configure_device(mixed_precision=MIXED_PRECISION)
print(f"\n✅ Menjalankan Pipeline pada Device: {device}")
print(f"📌 Versi TensorFlow: {tf.__version__}")

## 📦 Bagian 2: Extract ZIP Dataset & Inspection
Dataset utama berada di: `C:\Users\dzikri\Downloads\aware-project\_output_.zip` (~10GB). 
Kode di bawah ini akan mengekstrak file ZIP ke dalam direktori `extracted/` secara otomatis (jika belum terextract) dan melakukan inspeksi struktur folder serta jumlah gambar.

In [ ]:
from src.dataset import extract_dataset, inspect_dataset

# Ekstrak dataset (otomatis di-skip jika sudah ada)
extracted_root = extract_dataset(zip_path=ZIP_PATH, extract_to=EXTRACTED_DIR, force=False)

# Inspeksi isi dataset dan sebaran kelas
dataset_report = inspect_dataset(extracted_root)

## 🧹 Bagian 3: Data Preprocessing & Cleaning
Dataset citra mentah sering kali memiliki duplikat atau file yang rusak (corrupt). Proses `clean_dataset` akan memindai keseluruhan dataset dan memastikan integritas citra.

In [ ]:
from src.dataset import clean_dataset

# Pembersihan file duplikat (berbasis MD5 hash) & file corrupt
clean_stats = clean_dataset(extracted_root, remove_duplicates=True, remove_corrupt=True)

## 📊 Bagian 4: Label Analysis & Class Imbalance Strategy
Berdasarkan inspeksi, dataset `aware-v-02` memiliki 4 subfolder:
- `non_drowsy` (19,445 citra) & `no_yawning` (22,197 citra) $\rightarrow$ **Class 0 (Alert / Fit-to-Work)** [Total: 41,642 citra]
- `drowsy` (22,348 citra) & `yawning` (6,772 citra) $\rightarrow$ **Class 1 (Fatigue / Drowsy)** [Total: 29,120 citra]

Terdapat rasio ketidakseimbangan kelas (*imbalance ratio*) sekitar **1.4x** antar agregat biner, dan **3.3x** jika dilihat per kategori spesifik (`yawning` paling sedikit). Untuk mengatasi hal ini, kita menghitung **Class Weights** yang akan diberikan pada fungsi loss.

In [ ]:
from src.dataset import prepare_labels, split_dataset, compute_class_weights

# Pemetaan path citra ke label biner (0 = Alert, 1 = Fatigue)
image_paths, labels = prepare_labels(extracted_root, label_map=LABEL_MAP)

# Stratified splitting: 70% Train, 15% Validation, 15% Test
tr_split, val_split, te_split = split_dataset(image_paths, labels, train_ratio=TRAIN_SPLIT, val_ratio=VAL_SPLIT, seed=RANDOM_SEED)

# Perhitungan bobot kelas kompensasi imbalance
class_weights = compute_class_weights(tr_split[1])
print(f"\n⚖️ Computed Class Weights: {class_weights}")

## ⚡ Bagian 5: Data Augmentation & `tf.data` Pipeline
Untuk mencegah overfitting dan membuat model tahan terhadap berbagai kondisi pencahayaan di tempat kerja (office, pabrik, kabin kendaraan), kita menerapkan augmentasi tingkat lanjut (flip, random brightness, contrast, saturation, random zoom crop) secara efisien menggunakan `tf.data` pipeline.

In [ ]:
from src.dataset import build_pipeline

# Pembuatan tf.data.Dataset dengan pra-pemrosesan asinkron (AUTOTUNE)
ds_train = build_pipeline(tr_split[0],  tr_split[1],  augment=True,  shuffle=True,  batch_size=BATCH_SIZE)
ds_val   = build_pipeline(val_split[0], val_split[1], augment=False, shuffle=False, batch_size=BATCH_SIZE)
ds_test  = build_pipeline(te_split[0],  te_split[1],  augment=False, shuffle=False, batch_size=BATCH_SIZE)

print("✅ tf.data Pipeline berhasil dibangun.")

In [ ]:
# Visualisasi beberapa sampel batch dari training dataset
plt.figure(figsize=(12, 8))
for images, batch_labels in ds_train.take(1):
    for i in range(6):
        ax = plt.subplot(2, 3, i + 1)
        # Konversi citra [0, 1] float ke [0, 255] uint8 untuk plotting
        img_disp = (images[i].numpy() * 255).astype("uint8")
        plt.imshow(img_disp)
        lbl = "Fatigue (1)" if batch_labels[i].numpy() == 1 else "Alert (0)"
        plt.title(lbl, color="red" if lbl=="Fatigue (1)" else "green", fontweight="bold")
        plt.axis("off")
plt.tight_layout()
plt.show()

## 🧠 Bagian 6: Model Building (Subclassing + Custom Layer + Focal Loss)
Kita membangun arsitektur `AwareFatigueModel` yang menggabungkan MobileNetV2 dengan `ChannelAttention` (SE Block) dan `FatigueClassificationHead`.

In [ ]:
from src.model import build_model, FocalLoss

# Inisialisasi model
model = build_model(num_classes=NUM_CLASSES, dropout_rate=DROPOUT_RATE)

# Inisialisasi fungsi loss dengan gamma=2.0 dan label smoothing 0.1
loss_fn = FocalLoss(gamma=2.0, alpha=0.25, label_smoothing=LABEL_SMOOTHING)
print("\n✅ Model dan Custom Focal Loss siap digunakan.")

## 🔄 Bagian 7: Custom Training Loop dengan `tf.GradientTape` & Monitoring
Sesuai requirement tugas, proses training tidak menggunakan `model.fit()` standar, melainkan iterasi manual berbasis `tf.GradientTape` untuk mendapatkan kendali penuh atas perhitungan gradien, clipping, dan pembaruan bobot.

In [ ]:
from train import train_model

# Catatan: Untuk keperluan demonstrasi cepat di notebook, kita dapat mengatur jumlah epoch lebih sedikit
# Untuk training penuh hingga target Akurasi >= 85% dan MAE <= 0.02, gunakan EPOCHS_FROZEN=10 dan EPOCHS_FINETUNE=20
DEMO_EPOCHS_P1 = 2
DEMO_EPOCHS_P2 = 3

print("🚀 Mengawali Phase 1 Training (Backbone Frozen, Melatih Head & Attention Layer saja) ...")
model, p1_metrics = train_model(
    model, ds_train, ds_val, ds_test,
    tr_split[1], class_weights,
    epochs        = DEMO_EPOCHS_P1,  # ganti ke EPOCHS_FROZEN untuk full run
    learning_rate = LEARNING_RATE,
    phase         = "phase1"
)

In [ ]:
print("\n🚀 Mengawali Phase 2 Training (Fine-Tuning: Membuka kunci top layer backbone MobileNetV2) ...")
model.enable_fine_tuning(fine_tune_at=100)

model, p2_metrics = train_model(
    model, ds_train, ds_val, ds_test,
    tr_split[1], class_weights,
    epochs        = DEMO_EPOCHS_P2,  # ganti ke EPOCHS_FINETUNE untuk full run
    learning_rate = FINETUNE_LR,
    phase         = "phase2"
)

## 📈 Bagian 8: Evaluation on Test Set & Monitoring
Kita mengevaluasi performa akhir model menggunakan dataset held-out (test split 15%) dan memverifikasi pencapaian target:
- **Akurasi $\ge$ 85%**
- **MAE $\le$ 0.02**

In [ ]:
from train import evaluate_model

print("\n🔍 Menjalankan Evaluasi Akhir pada Test Set ...")
test_metrics = evaluate_model(model, ds_test)

## 💾 Bagian 9: Export Model (`.keras`, `SavedModel`, dan `TFLite`)
Model yang telah dilatih dan mencapai target akan diekspor untuk kebutuhan integrasi backend (misalnya menggunakan FastAPI di folder `api/`) maupun untuk perangkat edge.

In [ ]:
from train import export_model

print("\n💾 Menyimpan dan mengekspor model ke direktori exports/ ...")
export_model(model)
print("\n✅ Export selesai! Model siap digunakan di main.py / infer.py atau diintegrasikan ke server API.")

## 🖥️ Bagian 10: Inference Testing
Mari kita simulasikan proses prediksi (inference) pada sampel citra baru untuk melihat probabilitas dan keputusan akhir (Alert vs Fatigue).

In [ ]:
# Mengambil satu batch citra dari test dataset untuk pengujian inference
for test_imgs, test_lbls in ds_test.take(1):
    sample_img   = test_imgs[0:1] # Ambil 1 citra pertama
    actual_label = test_lbls[0].numpy()
    
    start_t   = time.time()
    prob_pred = model.predict_proba(sample_img)[0][0].numpy()
    inf_time  = (time.time() - start_t) * 1000 # ms
    
    pred_class = "Fatigue" if prob_pred >= 0.5 else "Alert"
    true_class = "Fatigue" if actual_label == 1 else "Alert"
    
    print(f"📌 Hasil Uji Inference Sample:")
    print(f"   - Probabilitas Fatigue : {prob_pred:.4f} ({prob_pred*100:.1f}%)")
    print(f"   - Prediksi Model       : {pred_class}")
    print(f"   - Ground Truth Label   : {true_class}")
    print(f"   - Waktu Komputasi      : {inf_time:.1f} ms")
    
    plt.figure(figsize=(4, 4))
    plt.imshow((sample_img[0].numpy() * 255).astype("uint8"))
    plt.title(f"Pred: {pred_class} | Actual: {true_class}", color="blue")
    plt.axis("off")
    plt.show()
    break

## 🚀 Bagian 11: Rekomendasi Improvement Lanjutan (Engineering Notes)

Untuk pengembangan ke tahap produksi berskala industri (*enterprise deployment*), berikut adalah eksperimen lanjutan yang direkomendasikan:
1. **Spatial-Temporal Modeling (Video Input)**: Menggabungkan CNN dengan **LSTM** atau **Video ViT (Vision Transformer)** untuk mendeteksi durasi penutupan kelopak mata (PERCLOS - *Percentage of Eye Closure*) secara berkala selama 3-5 detik.
2. **Mixed Precision & TensorRT**: Melakukan kuantisasi model ke INT8/FP16 menggunakan NVIDIA TensorRT atau TFLite XNNPACK agar waktu inferensi di edge device (Jetson Nano / Raspberry Pi) mencapai kurang dari 10 ms.
3. **Dynamic Attention Map (Grad-CAM)**: Menambahkan visualisasi Grad-CAM pada dashboard admin untuk memberikan *explainability* (penjelasan visual) mengapa sistem menyatakan seorang karyawan mengalami fatigue (misalnya *heatmap* menyorot area mata yang sembab atau menguap).
4. **Active Learning**: Mengimplementasikan *feedback loop* pada aplikasi di mana hasil deteksi dengan *confidence score* rendah ($0.45 < p < 0.55$) akan dikirim ke antrean kurasi manual untuk dilatih ulang (*retraining*).